In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import optuna
import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense


from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Adagrad, Adadelta, Nadam


from sklearn.metrics import roc_auc_score

In [ ]:
data = pd.read_csv('Employee.csv')

data

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1
...,...,...,...,...,...,...,...,...,...
4648,Bachelors,2013,Bangalore,3,26,Female,No,4,0
4649,Masters,2013,Pune,2,37,Male,No,2,1
4650,Masters,2018,New Delhi,3,27,Male,No,5,1
4651,Bachelors,2012,Bangalore,3,30,Male,Yes,2,0


In [ ]:
data.describe(include='all')

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
count,4653,4653.000000,4653,4653.000000,4653.000000,4653,4653,4653.000000,4653.000000
unique,3,NaN,3,NaN,NaN,2,2,NaN,NaN
top,Bachelors,NaN,Bangalore,NaN,NaN,Male,No,NaN,NaN
freq,3601,NaN,2228,NaN,NaN,2778,4175,NaN,NaN
mean,NaN,2015.062970,NaN,2.698259,29.393295,NaN,NaN,2.905652,0.343864
std,NaN,1.863377,NaN,0.561435,4.826087,NaN,NaN,1.558240,0.475047
min,NaN,2012.000000,NaN,1.000000,22.000000,NaN,NaN,0.000000,0.000000
25%,NaN,2013.000000,NaN,3.000000,26.000000,NaN,NaN,2.000000,0.000000
50%,NaN,2015.000000,NaN,3.000000,28.000000,NaN,NaN,3.000000,0.000000
75%,NaN,2017.000000,NaN,3.000000,32.000000,NaN,NaN,4.000000,1.000000


In [ ]:
data.isnull().sum()

,0
Education,0
JoiningYear,0
City,0
PaymentTier,0
Age,0
Gender,0
EverBenched,0
ExperienceInCurrentDomain,0
LeaveOrNot,0


In [ ]:
data = pd.get_dummies(data, drop_first=True)

data

,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot,Education_Masters,Education_PHD,City_New Delhi,City_Pune,Gender_Male,EverBenched_Yes
0,2017,3,34,0,0,False,False,False,False,True,False
1,2013,1,28,3,1,False,False,False,True,False,False
2,2014,3,38,2,0,False,False,True,False,False,False
3,2016,3,27,5,1,True,False,False,False,True,False
4,2017,3,24,2,1,True,False,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...
4648,2013,3,26,4,0,False,False,False,False,False,False
4649,2013,2,37,2,1,True,False,False,True,True,False
4650,2018,3,27,5,1,True,False,True,False,True,False
4651,2012,3,30,2,0,False,False,False,False,True,True


In [ ]:
targets = data['LeaveOrNot']

inputs = data.drop(['LeaveOrNot'],axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(inputs)

scaled = scaler.transform(inputs)

inputs_scaled = pd.DataFrame(scaled, columns=inputs.columns)

inputs_scaled

,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,Education_Masters,Education_PHD,City_New Delhi,City_Pune,Gender_Male,EverBenched_Yes
0,1.039638,0.537503,0.954645,-1.864901,-0.480575,-0.200022,-0.575282,-0.612041,0.821551,-0.338365
1,-1.107233,-3.025177,-0.288732,0.060554,-0.480575,-0.200022,-0.575282,1.633878,-1.217210,-0.338365
2,-0.570515,0.537503,1.783563,-0.581264,-0.480575,-0.200022,1.738277,-0.612041,-1.217210,-0.338365
3,0.502921,0.537503,-0.495961,1.344191,2.080840,-0.200022,-0.575282,-0.612041,0.821551,-0.338365
4,1.039638,0.537503,-1.117650,-0.581264,2.080840,-0.200022,-0.575282,1.633878,0.821551,2.955387
...,...,...,...,...,...,...,...,...,...,...
4648,-1.107233,0.537503,-0.703191,0.702373,-0.480575,-0.200022,-0.575282,-0.612041,-1.217210,-0.338365
4649,-1.107233,-1.243837,1.576334,-0.581264,2.080840,-0.200022,-0.575282,1.633878,0.821551,-0.338365
4650,1.576356,0.537503,-0.495961,1.344191,2.080840,-0.200022,1.738277,-0.612041,0.821551,-0.338365
4651,-1.643951,0.537503,0.125727,-0.581264,-0.480575,-0.200022,-0.575282,-0.612041,0.821551,2.955387


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(inputs_scaled, targets, test_size=0.2, random_state=42)

In [ ]:
def create_model(trial):
    model = Sequential()

    model.add(Dense(units=trial.suggest_int('units_layer1', 6, 32), activation='relu'))
    model.add(Dense(units=trial.suggest_int('units_layer2', 6, 32), activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))


    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop', 'adagrad'])
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)

    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        optimizer = SGD(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    elif optimizer_name == 'adagrad':
        optimizer = Adagrad(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['AUC'])

    return model

In [ ]:
def optimal(trial):


    epochs = trial.suggest_int('epochs', 10, 50)
    batch_size = trial.suggest_int('batch_size', 16, 64)

    model = create_model(trial)

    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size)

    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred)

    return auc

study = optuna.create_study(direction='maximize')
study.optimize(optimal, n_trials=10)

print(f"Best trial: {study.best_trial.value}")
print(f"Best hyperparameters: {study.best_trial.params}")

[I 2026-04-18 10:05:51,684] A new study created in memory with name: no-name-49222b9b-7118-4e37-a7e7-981f48b8b739


Epoch 1/19


/tmp/ipykernel_1867/3644950783.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


71/71 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - AUC: 0.5354 - loss: 0.6741
Epoch 2/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - AUC: 0.5353 - loss: 0.6740
Epoch 3/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - AUC: 0.5352 - loss: 0.6740
Epoch 4/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - AUC: 0.5353 - loss: 0.6739
Epoch 5/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - AUC: 0.5354 - loss: 0.6739
Epoch 6/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - AUC: 0.5353 - loss: 0.6739
Epoch 7/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - AUC: 0.5356 - loss: 0.6738
Epoch 8/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC: 0.5358 - loss: 0.6738
Epoch 9/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC: 0.5360 - loss: 0.6737
Epoch 10/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - AUC: 0.5360 - loss: 0.6737
Epoch 11/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - AUC: 0.5361 - loss: 0.6736
Epoch 12/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - AUC: 0.5361 - loss: 0.6736
Epoch 13/19
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC:

[I 2026-04-18 10:06:08,429] Trial 0 finished with value: 0.512568306010929 and parameters: {'epochs': 19, 'batch_size': 53, 'units_layer1': 22, 'units_layer2': 6, 'optimizer': 'sgd', 'learning_rate': 1.7995890543118157e-05}. Best is trial 0 with value: 0.512568306010929.


...

Epoch 1/48


/tmp/ipykernel_1867/3644950783.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6096 - loss: 0.6938
Epoch 2/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.6548 - loss: 0.6269
Epoch 3/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.6847 - loss: 0.6044
Epoch 4/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7064 - loss: 0.5899
Epoch 5/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7219 - loss: 0.5783
Epoch 6/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7310 - loss: 0.5686
Epoch 7/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7392 - loss: 0.5608
Epoch 8/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7453 - loss: 0.5543
Epoch 9/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7505 - loss: 0.5488
Epoch 10/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7547 - loss: 0.5443
Epoch 11/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7590 - loss: 0.5401
Epoch 12/48
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7629 - loss: 0.5364
Epoch 13/48
101/101 ━━━━━━━━━━━━━━━━

[I 2026-04-18 10:07:45,141] Trial 9 finished with value: 0.8567897451611255 and parameters: {'epochs': 48, 'batch_size': 37, 'units_layer1': 23, 'units_layer2': 30, 'optimizer': 'sgd', 'learning_rate': 0.008164707239111526}. Best is trial 6 with value: 0.8857106378632348.


Best trial: 0.8857106378632348
Best hyperparameters: {'epochs': 47, 'batch_size': 44, 'units_layer1': 10, 'units_layer2': 24, 'optimizer': 'rmsprop', 'learning_rate': 0.0013227354567160255}


In [ ]:
best_params = study.best_trial.params

best_params

{'epochs': 47,
 'batch_size': 44,
 'units_layer1': 10,
 'units_layer2': 24,
 'optimizer': 'rmsprop',
 'learning_rate': 0.0013227354567160255}

In [ ]:

best_model = Sequential()
best_model.add(Dense(units=best_params['units_layer1'], activation='relu'))
best_model.add(Dense(units=best_params['units_layer2'], activation='relu'))
best_model.add(Dense(1, activation='sigmoid'))

In [ ]:
if best_params['optimizer'] == 'adam':
    best_optimizer = Adam(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'sgd':
    best_optimizer = SGD(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'rmsprop':
    best_optimizer = RMSprop(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'adagrad':
    best_optimizer = Adagrad(learning_rate=best_params['learning_rate'])


In [ ]:
best_model.compile(optimizer=best_optimizer, loss='binary_crossentropy', metrics=['AUC'])

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train, epochs=47, batch_size=best_params['batch_size'])


    y_train_prob = model.predict(X_train)



    y_test_prob = model.predict(X_test)



    roc_train_prob = roc_auc_score(y_train, y_train_prob)
    gini_train_prob = roc_train_prob * 2 - 1



    roc_test_prob = roc_auc_score(y_test, y_test_prob)
    gini_test_prob = roc_test_prob * 2 - 1


    results = pd.DataFrame({
        'Dataset': ['Train', 'Test'],
        'Gini': [gini_train_prob * 100, gini_test_prob * 100],

    })

    return results

In [ ]:
evaluate(best_model, X_train, y_train, X_test, y_test)

Epoch 1/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - AUC: 0.7037 - loss: 0.5931
Epoch 2/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.7395 - loss: 0.5709
Epoch 3/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - AUC: 0.7593 - loss: 0.5519
Epoch 4/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - AUC: 0.7733 - loss: 0.5345
Epoch 5/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - AUC: 0.7844 - loss: 0.5191
Epoch 6/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - AUC: 0.7923 - loss: 0.5076
Epoch 7/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - AUC: 0.7986 - loss: 0.4973
Epoch 8/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - AUC: 0.8047 - loss: 0.4894
Epoch 9/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.8113 - loss: 0.4816
Epoch 10/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8149 - loss: 0.4756
Epoch 11/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.8199 - loss: 0.4696
Epoch 12/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.8258 - loss: 0.4628
Epoch 13/47
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/ste

,Dataset,Gini
0,Train,75.684992
1,Test,77.952607
